# Two Spark sessions: what is shared and what is not

`pipeline/validate copy.py` creates one session with `create_spark_session`, then two children with `spark.newSession()`:
`original_spark` (plain, temp views) and `changed_spark` (`apply_run_options` + `register_tables_as` from `config.toml`).
This notebook reproduces those exact calls and checks, one by one, what each session sees.

Spark splits a session's state in two:

| Part | Scope | Contains |
|---|---|---|
| `SparkContext` | **shared** by every session | driver JVM, executors, core config (`spark.master`, `spark.driver.memory`, ...), Spark UI |
| `SharedState` | **shared** by every session | external catalog (catalog **tables** / databases), `global_temp` views, `CacheManager` (cached data), static SQL config (`spark.sql.warehouse.dir`, ...) |
| `SessionState` | **per session** | runtime SQL config (`spark.sql.*` set with `conf.set`), **temp views**, current database, UDFs |

`newSession()` gives a fresh `SessionState` (its SQL config starts from the `SparkContext` config, **not** from the parent's runtime changes) and reuses the same `SparkContext` and `SharedState`.

In [ ]:
import sys

import pandas as pd

sys.path.insert(0, "../pipeline")
from config_loader import options
from constants import get_scenario_paths
from spark_session import (
    apply_run_options,
    create_spark_session,
    register_tables_in_spark,
)

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

SCALE_FACTOR = 1
_, parquet_dir, _ = get_scenario_paths(SCALE_FACTOR)
parquet_dir

## 1. Create the sessions exactly as `validate copy.py` does

In [ ]:
spark = create_spark_session(
    app_name="TPCDS_Validate",
    use_all_cores=options.spark.use_all_cores,
    tune_memory_allocation=options.spark.tune_memory_allocation
)
spark.sparkContext.setLogLevel("ERROR")

original_spark = spark.newSession()
changed_spark  = spark.newSession()
changed_configuration = apply_run_options(changed_spark, options)

sessions = {"spark": spark, "original": original_spark, "changed": changed_spark}

## 2. Same driver, same `SharedState`, different `SessionState`

Expected: one `SparkContext` / application id for all three; `SharedState` identical; `SessionState` different per session.

In [ ]:
def _same(a, b):
    return a.equals(b)

rows = []
for name, session in sessions.items():
    jsession = session._jsparkSession
    rows.append(
        {
            "session": name,
            "python_object_id": id(session),
            "spark_context_id": id(session.sparkContext),
            "application_id": session.sparkContext.applicationId,
            "ui_url": session.sparkContext.uiWebUrl,
            "shared_state_same_as_spark": _same(
                jsession.sharedState(), spark._jsparkSession.sharedState()
            ),
            "session_state_same_as_spark": _same(
                jsession.sessionState(), spark._jsparkSession.sessionState()
            ),
        }
    )
pd.DataFrame(rows)

## 3. Configuration

Compare a few keys across the three sessions:
- **core** keys (`spark.master`, `spark.driver.memory`, ...) belong to the `SparkContext` → same everywhere;
- **builder** `spark.sql.*` keys from `create_spark_session` (ANSI) were put in the `SparkContext` config when it was created → inherited by both children;
- **runtime** keys set by `apply_run_options` → only in `changed`; `original` keeps Spark's defaults (it does **not** copy anything from `changed`, nor from later `spark.conf.set` calls on the parent).

In [ ]:
CORE_KEYS = [
    "spark.master",
    "spark.app.name",
    "spark.driver.memory",
    "spark.driver.extraJavaOptions",
    "spark.memory.fraction",
    "spark.memory.storageFraction",
]
STATIC_SQL_KEYS = [
    "spark.sql.warehouse.dir",
    "spark.sql.catalogImplementation",
]
BUILDER_SQL_KEYS = [
    "spark.sql.ansi.enabled",
    "spark.sql.ansi.doubleQuotedIdentifiers",
]
RUN_OPTION_KEYS = [
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.autoBroadcastJoinThreshold",
    "spark.sql.shuffle.partitions",
    "spark.sql.cbo.enabled",
    "spark.sql.cbo.joinReorder.enabled",
]

def config_table(keys_by_kind):
    rows = []
    for kind, keys in keys_by_kind.items():
        for key in keys:
            values = {name: s.conf.get(key, None) for name, s in sessions.items()}
            rows.append(
                {
                    "kind": kind,
                    "key": key,
                    **values,
                    "original == changed": values["original"] == values["changed"],
                }
            )
    return pd.DataFrame(rows)

config_table(
    {
        "core": CORE_KEYS,
        "static sql": STATIC_SQL_KEYS,
        "builder sql": BUILDER_SQL_KEYS,
        "apply_run_options": RUN_OPTION_KEYS,
    }
)

### 3.1 Runtime config does not propagate after `newSession()`

Setting a runtime key on the parent now does not reach the children, and setting it on one child does not reach the other.

In [ ]:
KEY = "spark.sql.shuffle.partitions"
before = {name: s.conf.get(KEY) for name, s in sessions.items()}

spark.conf.set(KEY, "123")
original_spark.conf.set(KEY, "7")
after = {name: s.conf.get(KEY) for name, s in sessions.items()}

# Restore
spark.conf.unset(KEY)
original_spark.conf.unset(KEY)

pd.DataFrame({"before": before, "after setting parent=123, original=7": after})

### 3.2 Shared settings cannot be changed per session

Core and static SQL keys live in shared state, so Spark refuses to change them from a session (instead of silently changing them for everyone).

In [ ]:
for key, value in [
    ("spark.driver.memory", "1g"),
    ("spark.sql.warehouse.dir", "/tmp/other-warehouse"),
]:
    try:
        changed_spark.conf.set(key, value)
        print(f"{key}: set to {changed_spark.conf.get(key)} (accepted)")
    except Exception as exception:
        print(f"{key}: rejected -> {type(exception).__name__}: {str(exception).splitlines()[0]}")

## 4. Register the tables exactly as `validate copy.py` does

`original_spark` always gets temp views; `changed_spark` gets whatever `config.toml` says (`register_tables_as`, `cbo_enabled`, `enable_static_caching`).

In [ ]:
print(f"changed_spark registers as: {options.spark.register_tables_as!r}, "
      f"analyze={options.spark.cbo_enabled}, "
      f"static caching={options.spark.enable_static_caching}")

register_tables_in_spark(
    original_spark,
    parquet_dir,
    register_as="views",
    analyze=False,
    enable_static_caching=False
)
register_tables_in_spark(
    changed_spark,
    parquet_dir,
    register_as=options.spark.register_tables_as,
    analyze=options.spark.cbo_enabled,
    enable_static_caching=options.spark.enable_static_caching
)

## 5. What each session sees in its catalog

Expected:
- **temp views** (`isTemporary=True`) appear only in the session that created them;
- **catalog tables** (`isTemporary=False`, database `default`) are in the shared external catalog, so they appear in **all** sessions — including `spark` and `original_spark`, which never created them.

In [ ]:
def catalog_listing(session):
    return {
        (table.name, table.isTemporary)
        for table in session.catalog.listTables()
    }

listings = {name: catalog_listing(s) for name, s in sessions.items()}
all_entries = sorted(set().union(*listings.values()))
pd.DataFrame(
    [
        {
            "name": name,
            "kind": "temp view" if is_temporary else "catalog table",
            **{session: (name, is_temporary) in listings[session] for session in sessions},
        }
        for name, is_temporary in all_entries
    ]
)

## 6. What a table name resolves to in each session

When a name exists both as a temp view and as a catalog table, the **temp view wins** in that session. So, with `register_tables_as = "tables"`, `store_sales` in `original_spark` reads the temp view, in `changed_spark` the catalog table, and in `spark` (no temp views) also the catalog table.

The analyzed plan shows which one: a temp view appears as `SubqueryAlias store_sales` over a bare `Relation parquet`, a catalog table as `SubqueryAlias spark_catalog.default.store_sales`.

In [ ]:
TABLE = "store_sales"

def analyzed_plan(session, name):
    try:
        return session.table(name)._jdf.queryExecution().analyzed().toString()
    except Exception as exception:
        return f"ERROR: {type(exception).__name__}: {str(exception).splitlines()[0]}"

for name, session in sessions.items():
    print(f"=== {name}: {TABLE} ===")
    print(analyzed_plan(session, TABLE))
    print(f"=== {name}: spark_catalog.default.{TABLE} ===")
    print(analyzed_plan(session, f"spark_catalog.default.{TABLE}"))

### 6.1 Table statistics (CBO)

`ANALYZE TABLE` stores statistics in the shared catalog, attached to the **table**. The temp view in `original_spark` has none, so its plan only has a size estimate from the files — even though the data is the same.

In [ ]:
for name, session in sessions.items():
    stats = (
        session.table(TABLE)._jdf.queryExecution().optimizedPlan().stats().toString()
    )
    print(f"{name:>8}: {stats}")

## 7. Catalog tables are shared mutable state

`register_tables_in_spark(..., register_as="tables")` runs `DROP TABLE IF EXISTS` + `createTable`. Because the catalog is shared, a drop in one session drops the table for every session.
Demonstrated on a scratch table so the TPC-DS tables are not touched.

In [ ]:
import os

SCRATCH = "demo_shared_table"
scratch_path = os.path.join(parquet_dir, "item")

changed_spark.sql(f"DROP TABLE IF EXISTS {SCRATCH}")
changed_spark.catalog.createTable(SCRATCH, path=scratch_path, source="parquet")
print("created in changed, count from original:", original_spark.table(SCRATCH).count())

original_spark.sql(f"DROP TABLE {SCRATCH}")
print("dropped in original, exists in changed?", changed_spark.catalog.tableExists(SCRATCH))

### 7.1 Temp views are not (but `global_temp` views are)

In [ ]:
original_spark.range(3).createOrReplaceTempView("demo_temp_view")
original_spark.range(3).createOrReplaceGlobalTempView("demo_global_view")

print("temp view in original:", original_spark.catalog.tableExists("demo_temp_view"))
print("temp view in changed: ", changed_spark.catalog.tableExists("demo_temp_view"))
print("global view in changed:", changed_spark.table("global_temp.demo_global_view").count(), "rows")

original_spark.catalog.dropTempView("demo_temp_view")
original_spark.catalog.dropGlobalTempView("demo_global_view")

## 8. The cache is shared

The `CacheManager` lives in `SharedState` and matches cached data **by plan**, not by session or name. Caching a table in `changed_spark` therefore also serves any session whose query has the same plan — and `clearCache()` in any session empties it for all of them.

Expected: `spark` (which resolves `item` to the same catalog table) sees it cached. Whether `original_spark` does depends on whether its temp view's plan (`spark.read.parquet(path)`) matches the catalog table's plan — this cell shows the actual answer. If `isCached` is `True`, the plan with cached data contains `InMemoryRelation`.

In [ ]:
CACHED = "item"

def cache_status():
    rows = []
    for name, session in sessions.items():
        uses_cache = "InMemoryRelation" in (
            session.table(CACHED)._jdf.queryExecution().withCachedData().toString()
        )
        rows.append(
            {
                "session": name,
                "isCached": session.catalog.isCached(CACHED),
                "plan uses InMemoryRelation": uses_cache,
            }
        )
    return pd.DataFrame(rows)

changed_spark.catalog.cacheTable(CACHED)
changed_spark.table(CACHED).count()
print(f"after cacheTable({CACHED!r}) in changed:")
display(cache_status())

original_spark.catalog.clearCache()
print("after clearCache() in original:")
display(cache_status())

## 9. Same query, different plans

Both sessions run on the same data and the same executors, but plan with their own SQL config. With `adaptative_enabled = false` in `config.toml`, `changed` should have no `AdaptiveSparkPlan` root while `original` (Spark default, AQE on) does; the join strategies depend on `autoBroadcastJoinThreshold`.

In [ ]:
JOIN_SQL = '''
SELECT d_year, i_brand, SUM(ss_ext_sales_price) AS sales
FROM store_sales
JOIN date_dim ON ss_sold_date_sk = d_date_sk
JOIN item     ON ss_item_sk      = i_item_sk
GROUP BY d_year, i_brand
'''

for name in ("original", "changed"):
    print(f"=== {name} ===")
    sessions[name].sql(JOIN_SQL).explain(mode="simple")

## 10. Cleanup

In [ ]:
spark.stop()  # stops the shared SparkContext: every session is unusable after this

## Takeaways for `validate copy.py`

- **One JVM, one UI, one set of executors, one driver memory**: the two variants compete for the same resources; `use_all_cores` / `tune_memory_allocation` apply to both.
- **Runtime SQL config is isolated**: `apply_run_options(changed_spark, ...)` leaves `original_spark` on Spark defaults (plus the ANSI flags from the builder).
- **Temp views are isolated, catalog tables are shared**: with `register_tables_as = "tables"`, `original_spark` still reads its own temp views because they shadow the tables of the same name. If both sessions registered as `"tables"`, the second `register_tables_in_spark` would drop and recreate the first one's tables.
- **Statistics live on the catalog table**: only sessions that resolve to the table use them; the temp views in `original_spark` don't.
- **The cache is shared and matched by plan**: static caching in `changed_spark` can also speed up `original_spark` if the plans match (section 8), which would bias the comparison; `clearCache()` from either session clears both.